# CPU lease activation and gate contention audit

This self-contained diagnostic uses Python's standard library only. It embeds the exact `_PriorityGate` class from the frozen engineered notebook and the exact `NoActionLease` class from the pinned repository. It performs no model loading, HTTP requests, game actions, GPU operations, or competition submission.

**Scope:** synthetic token accounting and controlled one-slot contention. These results establish neither a gameplay score nor performance under 110 games, mixed priority snapshots, cache pressure, or a GPU backend. The 59,000-token trim interval is a hypothetical trace, not an observed invariant of the solver.

The executed outputs were captured by running each code cell with standard-library `compile`/`exec` in one shared namespace, in notebook order. No Jupyter kernel or nbclient was installed. You can also rerun the cells in a normal Python notebook.


In [1]:
import contextlib
import dataclasses
from dataclasses import dataclass
import hashlib
import heapq
import json
import logging
import threading
import time

PROVENANCE = {'source_commit': 'af8c037d600d9ec633bf1cdfa6c1a487e7c3311b', 'frozen_notebook': 'Notebook/arc3-target40-engineered.ipynb', 'frozen_notebook_sha256': '389871eb573c71bd54f6519aba450d2c00cecf194337b1ad580fb7a03ffc4078', 'gate_excerpt_sha256': 'de20ee282a1d147407c03322db1912af75245aaf73745499ebb99f0115b45a70', 'lease_module': 'scripts/no_action_handover.py', 'lease_module_sha256': '47a09f0bc40afcdb69db3823bbcf8f71794dbf796c899958266bedc92ba3e18c', 'lease_class_excerpt_sha256': '0c5bd52365b9d7928eb725909286701e98e09ab68947a4cd6925ec8a76576ae1'}
GATE_SOURCE = 'class _PriorityGate:\n    """Harness-side admission control, for when the server ignores `priority`.\n\n    The request field is a hint a backend may drop; this enforces the same\n    ordering locally by limiting how many games hold a slot at once and\n    choosing who gets a freed one.\n\n    A game keeps its slot while its prefix is intact, because interrupting it\n    there throws away a cache entry about to be reused. When the trimmer drops\n    a message the prefix is invalid anyway and the next request must be\n    prefilled from the divergence point - the one moment when yielding is free -\n    so that is where the handover happens.\n\n    Release and re-enqueue are one critical section. Releasing first would leave\n    a window in which a lower-priority waiter takes the slot the caller was\n    about to compete for.\n    """\n\n    def __init__(self, slots: int) -> None:\n        self._cond = threading.Condition()\n        self._free = max(1, int(slots))\n        self._waiting: list[tuple[int, int]] = []\n        self._seq = 0\n        self._admitted: set[int] = set()\n        self._snapshots: dict[int, PrioritySnapshot] = {}\n        self._clock_start: float | None = None\n        self._clock_deadline: float | None = None\n        self._tail_fade_announced = False\n\n    def configure_clock(self, started_at: float, deadline: float | None) -> None:\n        """One shared monotonic clock, configured before initial admission.\n\n        Earliest session start/deadline wins; appropriate for simultaneously\n        launched sessions. No clock is inferred from the last action payload.\n        """\n        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False) and deadline is None:\n            raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")\n        with self._cond:\n            self._clock_start = min(self._clock_start, started_at) if self._clock_start is not None else started_at\n            if deadline is not None:\n                self._clock_deadline = min(self._clock_deadline, deadline) if self._clock_deadline is not None else deadline\n\n    def _snapshot_priority(self, snapshot: PrioritySnapshot, now: float) -> int:\n        elapsed = max(0.0, now - self._clock_start) if self._clock_start is not None else 0.0\n        endgame = _endgame_start_minutes() > 0 and elapsed >= 60 * _endgame_start_minutes()\n        tail_fraction = None\n        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False):\n            if self._clock_deadline is None or self._clock_start is None:\n                raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")\n            fraction_setting = os.environ.get("ARC3_PRIORITY_TAIL_FADE_FRACTION", "").strip()\n            if fraction_setting:\n                try:\n                    fraction = float(fraction_setting)\n                except ValueError:\n                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1") from None\n                if not 0.0 < fraction <= 1.0:\n                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1")\n                # Use the full shared run duration, not the shrinking time left.\n                window = fraction * max(0.0, self._clock_deadline - self._clock_start)\n            else:\n                window = max(1.0, 60 * _get_env_float("ARC3_PRIORITY_TAIL_FADE_MINUTES", 90.0))\n            remaining = max(0.0, self._clock_deadline - now)\n            tail_fraction = remaining / window if window > 0.0 else 0.0\n            if remaining <= window:\n                # Snapshot scoring also happens outside _pump\'s lock. Announce\n                # once per shared gate, even when several games trim together.\n                with self._cond:\n                    if not self._tail_fade_announced:\n                        self._tail_fade_announced = True\n                        log.warning(\n                            "tail fade phase after %.1f minutes: %.1f minutes "\n                            "remaining; fading tail B to zero over the final "\n                            "%.1f minutes, retaining hazard C",\n                            elapsed / 60.0, remaining / 60.0, window / 60.0,\n                        )\n        return priority_value(\n            snapshot, endgame=endgame, tail_fraction=tail_fraction,\n            human_actions=_priority_human_actions(), action_scale=_priority_action_scale(),\n            token_scale=_priority_token_scale(), action_weight=_priority_action_weight(),\n            tail_base=_priority_tail_base(), tail_cap=_priority_tail_cap(),\n            **_priority_level_options(),\n        )\n\n    def acquire(self, priority: int) -> None:\n        with self._cond:\n            self._enqueue_and_wait(priority)\n\n    def release(self) -> None:\n        with self._cond:\n            self._free += 1\n            self._pump()\n\n    def handover(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:  # noqa: D401\n        """Give up the slot and immediately compete for it again.\n\n        The caller may win it straight back, which is the right outcome when\n        nothing else is waiting or everything else is further along.\n        """\n        _t0 = time.monotonic()\n        with self._cond:\n            self._free += 1\n            self._enqueue_and_wait(priority, snapshot)\n        _GATE_STATS["handover_ms"] += int(1000 * (time.monotonic() - _t0))\n\n    def _enqueue_and_wait(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:\n        # TEMPORARY: with slots == games nobody should ever block here\n        _GATE_STATS["enqueued"] += 1\n        self._seq += 1\n        token = self._seq\n        if snapshot is not None:\n            self._snapshots[token] = snapshot\n        # negated so the heap pops the HIGHEST priority; the sequence number\n        # breaks ties by arrival\n        heapq.heappush(self._waiting, (-int(priority), token))\n        self._pump()\n        if token not in self._admitted:\n            _GATE_STATS["blocked"] += 1\n            t0 = time.monotonic()\n            while token not in self._admitted:\n                self._cond.wait()\n            _GATE_STATS["blocked_ms"] += int(1000 * (time.monotonic() - t0))\n        self._admitted.discard(token)\n\n    def _pump(self) -> None:\n        if _priority_refresh_enabled() and self._free > 0:\n            now = time.monotonic()\n            self._waiting = [\n                (-self._snapshot_priority(self._snapshots[token], now), token)\n                if token in self._snapshots else (priority, token)\n                for priority, token in self._waiting\n            ]\n            heapq.heapify(self._waiting)\n        woke = False\n        while self._free > 0 and self._waiting:\n            _, token = heapq.heappop(self._waiting)\n            self._snapshots.pop(token, None)\n            self._admitted.add(token)\n            self._free -= 1\n            woke = True\n        if woke:\n            self._cond.notify_all()\n\n'
LEASE_SOURCE = '@dataclass\nclass NoActionLease:\n    """Per-agent accounting; does not reset on analyzer yield/resumption."""\n\n    quantum: int\n    last_actions: int | None = None\n    last_tokens: int = 0\n    action_anchor: int = 0\n    lease_anchor: int = 0\n\n    def observe(self, tokens: int, actions: int, *, ordinary_handover=False) -> bool:\n        if tokens < 0 or actions < 0:\n            raise ValueError("Token and action counts must be nonnegative")\n        if (self.last_actions is None or actions != self.last_actions\n                or tokens < self.last_tokens):\n            # Action progress, a session reset, or existing request-error token\n            # rollback begins a new lease. Never turn a negative delta into a\n            # false handover. Tokens spent reaching an action are behind us.\n            self.action_anchor = self.lease_anchor = tokens\n        self.last_actions = actions\n        self.last_tokens = tokens\n        if ordinary_handover:\n            self.lease_anchor = tokens\n        return self.quantum > 0 and tokens - self.lease_anchor >= self.quantum\n\n    def handed_over(self, tokens: int) -> None:\n        self.lease_anchor = tokens\n'
assert hashlib.sha256(GATE_SOURCE.encode()).hexdigest() == PROVENANCE["gate_excerpt_sha256"]
assert hashlib.sha256(LEASE_SOURCE.encode()).hexdigest() == PROVENANCE["lease_class_excerpt_sha256"]
exec(compile(LEASE_SOURCE, "pinned_no_action_lease_excerpt.py", "exec"), globals())
print(json.dumps(PROVENANCE, indent=2))


{
  "source_commit": "af8c037d600d9ec633bf1cdfa6c1a487e7c3311b",
  "frozen_notebook": "Notebook/arc3-target40-engineered.ipynb",
  "frozen_notebook_sha256": "389871eb573c71bd54f6519aba450d2c00cecf194337b1ad580fb7a03ffc4078",
  "gate_excerpt_sha256": "de20ee282a1d147407c03322db1912af75245aaf73745499ebb99f0115b45a70",
  "lease_module": "scripts/no_action_handover.py",
  "lease_module_sha256": "47a09f0bc40afcdb69db3823bbcf8f71794dbf796c899958266bedc92ba3e18c",
  "lease_class_excerpt_sha256": "0c5bd52365b9d7928eb725909286701e98e09ab68947a4cd6925ec8a76576ae1"
}


## Lease activation

All traces hold the action count constant and initialize accounting at zero. A completed voluntary handover moves the lease anchor using the real `handed_over` method. The trim trace calls `observe(..., ordinary_handover=True)` every 59,000 tokens. Observe points include every 1,024-token boundary, exact trim boundaries, and the final token count.


In [2]:
def lease_trace(trim_interval=None, quantum=65536, final_tokens=262144):
    lease = NoActionLease(quantum)
    lease.observe(0, 0)
    trims = set(range(trim_interval, final_tokens + 1, trim_interval)) if trim_interval else set()
    checkpoints = sorted(set(range(1024, final_tokens + 1, 1024)) | trims | {final_tokens})
    events = []
    for tokens in checkpoints:
        ordinary = tokens in trims
        due = lease.observe(tokens, 0, ordinary_handover=ordinary)
        if ordinary:
            events.append({"tokens": tokens, "kind": "ordinary_handover"})
        if due:
            events.append({"tokens": tokens, "kind": "voluntary_handover"})
            lease.handed_over(tokens)
    return {"quantum": quantum, "hypothetical_trim_interval": trim_interval,
            "final_tokens": final_tokens, "actions": 0,
            "ordinary_handovers": sum(e["kind"] == "ordinary_handover" for e in events),
            "voluntary_handovers": sum(e["kind"] == "voluntary_handover" for e in events),
            "events": events}

TRACE_NO_TRIMS = lease_trace()
TRACE_59000_TRIMS = lease_trace(59000)
assert TRACE_NO_TRIMS["voluntary_handovers"] == 4
assert TRACE_NO_TRIMS["ordinary_handovers"] == 0
assert TRACE_59000_TRIMS["voluntary_handovers"] == 0
assert TRACE_59000_TRIMS["ordinary_handovers"] == 4
print(json.dumps({"without_trims": TRACE_NO_TRIMS, "with_59000_trims": TRACE_59000_TRIMS}, indent=2))


{
  "without_trims": {
    "quantum": 65536,
    "hypothetical_trim_interval": null,
    "final_tokens": 262144,
    "actions": 0,
    "ordinary_handovers": 0,
    "voluntary_handovers": 4,
    "events": [
      {
        "tokens": 65536,
        "kind": "voluntary_handover"
      },
      {
        "tokens": 131072,
        "kind": "voluntary_handover"
      },
      {
        "tokens": 196608,
        "kind": "voluntary_handover"
      },
      {
        "tokens": 262144,
        "kind": "voluntary_handover"
      }
    ]
  },
  "with_59000_trims": {
    "quantum": 65536,
    "hypothetical_trim_interval": 59000,
    "final_tokens": 262144,
    "actions": 0,
    "ordinary_handovers": 4,
    "voluntary_handovers": 0,
    "events": [
      {
        "tokens": 59000,
        "kind": "ordinary_handover"
      },
      {
        "tokens": 118000,
        "kind": "ordinary_handover"
      },
      {
        "tokens": 177000,
        "kind": "ordinary_handover"
      },
      {
        "toke

## Real gate, queued contender and distinct owners

The pinned class is executed unchanged. Only its surrounding namespace is supplied: standard-library imports, disabled tail-fade timing, and the original gate counters. No refreshed priority snapshots are submitted in this fixture; `PrioritySnapshot` is therefore an unused annotation binding, and the scoring helpers cannot affect admission. Priorities are explicit integers.

Owner A holds the sole slot. Owner B is confirmed in the real `_waiting` queue before A crosses the 65,536-token lease threshold and calls the real `handover`. At equal priority the earlier queued contender should acquire first. At higher A priority, immediate reacquisition is an expected counterexample: calling `handover` alone does not prove a distinct owner gained the slot. Threads use bounded waits and daemon fallback; cleanup admits blocked waiters on failure so the diagnostic cannot hang indefinitely.


In [3]:
def real_gate_scope():
    scope = dict(threading=threading, time=time, heapq=heapq,
                 PrioritySnapshot=object,
                 _priority_refresh_enabled=lambda: True,
                 _get_env_bool=lambda key, default=False: False,
                 _GATE_STATS=dict(enqueued=0, blocked=0, blocked_ms=0, handover_ms=0),
                 log=logging.getLogger("cpu_gate_audit"))
    exec(compile(GATE_SOURCE, "pinned_real_priority_gate.py", "exec"), scope)
    return scope

def contention_case(owner_priority, contender_priority, timeout=3.0):
    scope = real_gate_scope()
    gate = scope["_PriorityGate"](1)
    owner_acquired = threading.Event()
    begin_handover = threading.Event()
    owner_resumed = threading.Event()
    contender_acquired = threading.Event()
    release_contender = threading.Event()
    trace_lock = threading.Lock()
    trace, errors = [], []
    epoch = time.monotonic()
    def record(owner, event, **details):
        with trace_lock:
            trace.append(dict(owner=owner, event=event,
                              elapsed_seconds=round(time.monotonic() - epoch, 6), **details))
    def owner():
        holding = False
        try:
            gate.acquire(owner_priority); holding = True
            record("A", "acquired")
            owner_acquired.set()
            if not begin_handover.wait(timeout):
                raise TimeoutError("owner was not released to hand over")
            lease = NoActionLease(65536)
            lease.observe(0, 0)
            due = lease.observe(65536, 0)
            if not due:
                raise AssertionError("real lease did not activate")
            with gate._cond:
                queued = len(gate._waiting)
            record("A", "lease_due", generated_tokens=65536, queued_contenders=queued)
            gate.handover(owner_priority)
            lease.handed_over(65536)
            record("A", "handover_returned")
            owner_resumed.set()
        except BaseException as exc:
            errors.append("owner: " + repr(exc))
        finally:
            if holding:
                gate.release()
    def contender():
        holding = False
        try:
            gate.acquire(contender_priority); holding = True
            record("B", "acquired")
            contender_acquired.set()
            if not release_contender.wait(timeout):
                raise TimeoutError("contender was not released")
        except BaseException as exc:
            errors.append("contender: " + repr(exc))
        finally:
            if holding:
                record("B", "released")
                gate.release()
    threads = [threading.Thread(target=owner, daemon=True),
               threading.Thread(target=contender, daemon=True)]
    try:
        threads[0].start()
        if not owner_acquired.wait(timeout):
            raise TimeoutError("initial owner did not acquire")
        threads[1].start()
        deadline = time.monotonic() + timeout
        while True:
            with gate._cond:
                queued_before = len(gate._waiting)
            if queued_before == 1:
                break
            if time.monotonic() >= deadline:
                raise TimeoutError("contender did not enter real queue")
            contender_acquired.wait(0.001)
        begin_handover.set()
        if owner_priority == contender_priority:
            if not contender_acquired.wait(timeout):
                raise TimeoutError("equal-priority contender did not acquire")
            resumed_while_contender_holds = owner_resumed.is_set()
            release_contender.set()
            if not owner_resumed.wait(timeout):
                raise TimeoutError("owner did not resume after contender release")
        else:
            if not owner_resumed.wait(timeout):
                raise TimeoutError("higher-priority owner did not resume")
            if not contender_acquired.wait(timeout):
                raise TimeoutError("contender did not acquire after owner release")
            resumed_while_contender_holds = None
            release_contender.set()
    except BaseException as exc:
        errors.append("controller: " + repr(exc))
    finally:
        begin_handover.set(); release_contender.set()
        for thread in threads:
            if thread.ident is not None:
                thread.join(timeout)
        if any(thread.is_alive() for thread in threads):
            # Diagnostic-only failure cleanup; never used in the successful path.
            with gate._cond:
                gate._free += 2
                gate._pump()
            for thread in threads:
                if thread.ident is not None:
                    thread.join(timeout)
        alive = sum(thread.is_alive() for thread in threads)
    acquisitions = [e["owner"] for e in trace if e["event"] in ("acquired", "handover_returned")]
    with gate._cond:
        final_free, final_waiting = gate._free, len(gate._waiting)
    return {"owner_priority": owner_priority, "contender_priority": contender_priority,
            "queued_before_handover": locals().get("queued_before", 0),
            "owner_sequence": acquisitions,
            "distinct_owner_transitions": sum(a != b for a, b in zip(acquisitions, acquisitions[1:])),
            "owner_resumed_while_contender_holds": locals().get("resumed_while_contender_holds"),
            "threads_alive": alive, "final_free_slots": final_free,
            "final_waiting": final_waiting, "gate_counters": dict(scope["_GATE_STATS"]),
            "errors": errors, "events": trace}

EQUAL_PRIORITY_RUNS = [contention_case(100, 100) for _ in range(3)]
HIGHER_OWNER_PRIORITY = contention_case(101, 100)
for result in EQUAL_PRIORITY_RUNS:
    assert not result["errors"], result
    assert result["owner_sequence"] == ["A", "B", "A"], result
    assert result["queued_before_handover"] == 1
    assert result["owner_resumed_while_contender_holds"] is False
    assert result["threads_alive"] == 0
    assert result["final_free_slots"] == 1 and result["final_waiting"] == 0
assert not HIGHER_OWNER_PRIORITY["errors"], HIGHER_OWNER_PRIORITY
assert HIGHER_OWNER_PRIORITY["owner_sequence"] == ["A", "A", "B"], HIGHER_OWNER_PRIORITY
assert HIGHER_OWNER_PRIORITY["threads_alive"] == 0
assert HIGHER_OWNER_PRIORITY["final_free_slots"] == 1
assert HIGHER_OWNER_PRIORITY["final_waiting"] == 0
print(json.dumps({"equal_priority_three_runs": EQUAL_PRIORITY_RUNS,
                  "higher_owner_priority_counterexample": HIGHER_OWNER_PRIORITY}, indent=2))


{
  "equal_priority_three_runs": [
    {
      "owner_priority": 100,
      "contender_priority": 100,
      "queued_before_handover": 1,
      "owner_sequence": [
        "A",
        "B",
        "A"
      ],
      "distinct_owner_transitions": 2,
      "owner_resumed_while_contender_holds": false,
      "threads_alive": 0,
      "final_free_slots": 1,
      "final_waiting": 0,
      "gate_counters": {
        "enqueued": 3,
        "blocked": 2,
        "blocked_ms": 0,
        "handover_ms": 0
      },
      "errors": [],
      "events": [
        {
          "owner": "A",
          "event": "acquired",
          "elapsed_seconds": 0.003568
        },
        {
          "owner": "A",
          "event": "lease_due",
          "elapsed_seconds": 0.004817,
          "generated_tokens": 65536,
          "queued_contenders": 1
        },
        {
          "owner": "B",
          "event": "acquired",
          "elapsed_seconds": 0.0049
        },
        {
          "owner": "B",
    

In [4]:
SUMMARY = {
    "execution": "Notebook Python cells; local capture uses stdlib compile/exec in shared namespace",
    "gpu_operations": 0,
    "model_loads": 0,
    "network_requests": 0,
    "lease_without_trims_voluntary_handovers": TRACE_NO_TRIMS["voluntary_handovers"],
    "lease_with_hypothetical_59000_trims_voluntary_handovers": TRACE_59000_TRIMS["voluntary_handovers"],
    "equal_priority_owner_sequences": [r["owner_sequence"] for r in EQUAL_PRIORITY_RUNS],
    "higher_priority_owner_sequence": HIGHER_OWNER_PRIORITY["owner_sequence"],
    "interpretation": [
        "The current 65536 lease can activate without earlier anchor resets.",
        "The same lease remains inactive in the hypothetical trace with an ordinary handover every59000 tokens.",
        "The real one-slot gate transfers to a queued equal-priority contender in all three controlled runs.",
        "A higher-priority caller can reacquire immediately; a handover count does not establish distinct-owner rotation.",
        "Actual solver trim cadence, refreshed D-prime priorities, cache cost, gameplay quality and score remain unmeasured."
    ],
    "score_evidence": False,
}
from pathlib import Path
RESULT_PATH = Path("arc3_cpu_lease_contention_audit.json")
RESULT_PATH.write_text(json.dumps(SUMMARY, indent=2) + "\n")
print(json.dumps(SUMMARY, indent=2))
print("Saved synthetic result:", str(RESULT_PATH))


{
  "execution": "Notebook Python cells; local capture uses stdlib compile/exec in shared namespace",
  "gpu_operations": 0,
  "model_loads": 0,
  "network_requests": 0,
  "lease_without_trims_voluntary_handovers": 4,
  "lease_with_hypothetical_59000_trims_voluntary_handovers": 0,
  "equal_priority_owner_sequences": [
    [
      "A",
      "B",
      "A"
    ],
    [
      "A",
      "B",
      "A"
    ],
    [
      "A",
      "B",
      "A"
    ]
  ],
  "higher_priority_owner_sequence": [
    "A",
    "A",
    "B"
  ],
  "interpretation": [
    "The current 65536 lease can activate without earlier anchor resets.",
    "The same lease remains inactive in the hypothetical trace with an ordinary handover every59000 tokens.",
    "The real one-slot gate transfers to a queued equal-priority contender in all three controlled runs.",
    "A higher-priority caller can reacquire immediately; a handover count does not establish distinct-owner rotation.",
    "Actual solver trim cadence, refre

## Decision

This fixture closes a CPU-level gap: it observes an actual queued contender and distinct ownership transfer in the original gate, and it shows why voluntary handover counts alone are insufficient. It also demonstrates the exact lease-reset mechanism under an explicit hypothetical trim schedule.

It does **not** justify changing the quantum or claiming a score increase. Before another full GPU gameplay evaluation, record real trim intervals, queued contenders, distinct owner changes, token/cache accounting and elapsed request time together under the intended policy. Preserve the distinction between this synthetic audit, a public gameplay evaluation and an official hidden competition score.
